# K-SPADA Data Preparation

**Audience:** users preparing spatial transcriptomics data for K-SPADA.  
**Prerequisites:** an AnnData file containing `x`, `y`, `batch`, and `Celltype`.  
**Goal:** create separate train/validation and query communication-graph files without expression binning.

## Outline
1. Load and inspect the source dataset.
2. Define sample-level train, validation, and query batches.
3. Split the AnnData object by batch.
4. Calculate pathway scores and communication graphs.
5. Assemble the final train/validation file.

In [ ]:
from pathlib import Path
import anndata as ad
import pandas as pd
import KSPADA

PROJECT_ROOT = Path('/home/lixiangyu/sh/program/K-SPADA')
DATA_DIR = Path('/home/lixiangyu/TOSICA-main/data/liver_cancer')
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'tutorial_data_preparation'
SOURCE_H5AD = DATA_DIR / 'all_hvg_3000.h5ad'
FORCE = True

TRAIN_BATCHES = ['1', '2']
VALIDATION_BATCHES = ['3']
QUERY_BATCHES = ['4']
training_config = KSPADA.BEST_LIVER_CANCER_TRAINING
CGRAPH_K = training_config.communication_neighbors

print('Source:', SOURCE_H5AD)


## Step 1 — Inspect the source data

The tutorial keeps complete biological batches separate; cells are never randomly split across subsets.

In [ ]:
source_adata = ad.read_h5ad(SOURCE_H5AD, backed='r')
required_columns = {'x', 'y', 'batch', 'Celltype'}
missing_columns = required_columns.difference(source_adata.obs.columns)
if missing_columns:
    raise KeyError(f'Missing required obs columns: {sorted(missing_columns)}')

dataset_summary = pd.Series({
    'cells': source_adata.n_obs,
    'genes': source_adata.n_vars,
    'batches': source_adata.obs['batch'].astype(str).nunique(),
    'cell_types': source_adata.obs['Celltype'].astype(str).nunique(),
})
display(dataset_summary)
source_adata.file.close()

## Step 2 — Create train/validation and query AnnData objects

In [ ]:
all_required_batches = set(TRAIN_BATCHES + VALIDATION_BATCHES + QUERY_BATCHES)
full_adata = ad.read_h5ad(SOURCE_H5AD)
full_adata.obs['batch'] = full_adata.obs['batch'].astype(str)
missing_batches = all_required_batches.difference(full_adata.obs['batch'].unique())
if missing_batches:
    raise ValueError(f'Missing requested batches: {sorted(missing_batches)}')

train_valid_adata = full_adata[full_adata.obs['batch'].isin(TRAIN_BATCHES + VALIDATION_BATCHES)].copy()
query_adata = full_adata[full_adata.obs['batch'].isin(QUERY_BATCHES)].copy()
print('Train/validation shape:', train_valid_adata.shape)
print('Query shape:', query_adata.shape)

## Step 3 — Build pathway scores and communication graphs

Write the split AnnData objects, calculate pathway activity scores, and construct the communication graphs.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_valid_source = OUTPUT_DIR / 'liver_train_valid_source.h5ad'
query_source = OUTPUT_DIR / 'liver_query_source.h5ad'
KSPADA.write_adata(train_valid_adata, train_valid_source, force=FORCE)
KSPADA.write_adata(query_adata, query_source, force=FORCE)

prepared_paths = KSPADA.prepare_data_workflow(
    train_h5ad=train_valid_source,
    query_h5ad=query_source,
    output_dir=OUTPUT_DIR,
    train_batches=TRAIN_BATCHES,
    valid_batches=VALIDATION_BATCHES,
    query_batches=QUERY_BATCHES,
    cgraph_k=CGRAPH_K,
    force=FORCE,
)
display(pd.Series({key: str(value) for key, value in prepared_paths.items() if not isinstance(value, dict)}))


## Exercise

Replace batch `4` with another held-out batch and verify that the train, validation, and query lists are disjoint.

In [ ]:
exercise_train = ['1', '2']
exercise_valid = ['3']
exercise_query = ['5']
assert not set(exercise_train) & set(exercise_valid)
assert not set(exercise_train) & set(exercise_query)
assert not set(exercise_valid) & set(exercise_query)

## Pitfall and extension

**Pitfall:** do not normalize an already normalized HVG matrix a second time.  
**Extension:** repeat the workflow with another tissue while preserving the same `x`, `y`, `batch`, and `Celltype` contract.